In [1]:
import polars as pl
import pandas as pd
import numpy as np
import pickle as pkl

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
import catboost as cb
from processor import PolarsLoader, ExprProcessor, PandasConverter
from IPython.display import Markdown

In [3]:
from sklearn.metrics import roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, ShuffleSplit, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder

skf = StratifiedKFold(3, random_state = 123, shuffle = True)
ss = StratifiedShuffleSplit(1, train_size = 0.8, random_state = 123)
ss_v = StratifiedShuffleSplit(1, train_size = 0.9, random_state = 123)

In [4]:
p = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor({
        'loan_paid_back': pl.col('loan_paid_back').cast(pl.Int8)
    }),
    PandasConverter(index_col = 'id')
)
df_train = p.fit_transform('data/train.csv')
df_test = p.transform('data/test.csv')
with open('grade_subgrade.pkl', 'rb') as f:
    c_map = pkl.load(f)
df_train['grade_subgrade_no'] = df_train['grade_subgrade'].map(c_map).astype('int')
df_test['grade_subgrade_no'] = df_test['grade_subgrade'].map(c_map).astype('int')
df_train.shape, df_test.shape

((593994, 13), (254569, 12))

In [5]:
X_all = df_test.columns.tolist()
X_num = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate', 'grade_subgrade_no']
X_cat = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
y = 'loan_paid_back'

In [6]:
import importlib
from modeler import Experimenter

/home/sun9sun9/jnote/sunkusun9/kaggle/PGS5/PGS5_ep11/modeler/_node.py:184: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  elif self.status is "finalized":


In [7]:
e = Experimenter(df_train.sample(frac = 0.01, random_state = 123), 'exp', sp = skf, sp_v = ss_v, splitter_params = {'y': y})

In [8]:
Markdown(
    e.desc_spec()
)

| 항목 | 값 |
|------|-----|
| **Outer Splitter (sp)** | `StratifiedKFold(n_splits=3, random_state=123, shuffle=True)` |
| **Inner Splitter (sp_v)** | `StratifiedShuffleSplit(n_splits=1, random_state=123)` |
| **Splitter Params** | `{y='loan_paid_back'}` |
| **Outer Folds** | 3 |
| **Inner Folds** | 1 |

In [9]:
e.add_grp('clf', 'exp', parent_grp = None, edges = [(None, [y])], y = y, method = 'predict_proba')
e.add_grp('preprocessor', 'pipe', parent_grp = None, method = 'transform')

In [10]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])
e.set_node('ohe', 'preprocessor', OneHotEncoder, edges = [(None, X_cat)], params={'sparse_output': False})
e.build()

🔄 Building 2 node(s)
  ├─ Building 'ohe'...
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'std'...
🔄 Building 2 node(s)
✅ Build complete!


In [11]:
e.rename_grp('preprocessor', 'preproc')

In [12]:
e.rename_grp('preproc', 'preprocessor')

In [13]:
e.build()

🔄 Building 0 node(s)
🔄 Building 0 node(s)
✅ Build complete!


In [14]:
e.build(rebuild=True)

🔄 Building 2 node(s)
  ├─ Building 'ohe'...
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'std'...
  ├─ Building 'ohe'...
  ├─ Building 'std'...
🔄 Building 2 node(s)
✅ Build complete!


In [15]:
from sklearn.linear_model import LogisticRegression

e.add_grp('lr', parent_grp = 'clf', processor = LogisticRegression)

In [16]:
from modeler import col
e.set_node('lr1', 'lr', edges = [('std', None)])
e.set_node('lr2', 'lr', edges = [('std', None), ('ohe', col.ohe_drop_first)])

In [17]:
e.build()

🔄 Building 0 node(s)
🔄 Building 0 node(s)
✅ Build complete!


In [18]:
results = e.nodes['lr1'].experiment(0, ['output'])
results

<generator object Node.experiment at 0x7f57de769a20>

In [19]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["2 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

In [20]:
Markdown(
    e.desc_node('lr2', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr2["lr2"]
        lr2_dummy[ ]
        style lr2_dummy fill:none,stroke:none
    end
    style node_lr2 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_dummy[ ]
        style ohe_dummy fill:none,stroke:none
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_dummy[ ]
        style std_dummy fill:none,stroke:none
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr2
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr2
    node_std --> node_lr2
```

**Path from Root to 'lr2' (3 path(s) found)**

In [21]:
e.add_grp('dim_reduction', parent_grp = 'preprocessor')

In [22]:
from sklearn.decomposition import PCA
e.set_node('pca', 'dim_reduction', processor=PCA, edges = [('std', None)], params={'n_components': 0.9})

In [23]:
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])
e.build()

  └─ Effeced 3 dependent node(s): ['lr1', 'lr2', 'pca']
🔄 Building 1 node(s)
  ├─ Building 'std'...
  ├─ Building 'std'...
  ├─ Building 'std'...
🔄 Building 1 node(s)
✅ Build complete!


In [24]:
e.exp('lr*', retry=True)

🔄 Experimenting 2 node(s)
0 fold
  ├─ Experimenting 'lr1'...
  ├─ Experimenting 'lr2'...
1 fold
  ├─ Experimenting 'lr1'...
  ├─ Experimenting 'lr2'...
2 fold
  ├─ Experimenting 'lr1'...
  ├─ Experimenting 'lr2'...
✅ Experimentation complete!


In [25]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph grp_preprocessor["preprocessor"]
        node_std["std"]
        style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        node_ohe["ohe"]
        style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px
        subgraph grp_dim_reduction["dim_reduction"]
            grp_dim_reduction_count["1 node(s)"]
            style grp_dim_reduction_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_dim_reduction fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_preprocessor fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    subgraph grp_clf["clf"]
        subgraph grp_lr["lr"]
            grp_lr_count["2 node(s)"]
            style grp_lr_count fill:#f5f5f5,stroke:#9e9e9e,stroke-dasharray: 5 5
        end
        style grp_lr fill:#e3f2fd,stroke:#1976d2,stroke-width:2px
    end
    style grp_clf fill:#e3f2fd,stroke:#1976d2,stroke-width:2px

    Root --> grp_clf
    Root --> grp_preprocessor
    grp_preprocessor --> grp_clf
```

In [26]:
e.set_node('lr3', 'lr', edges = [('ohe', None), ('pca', None)])

In [27]:
Markdown(
    e.desc_node('lr3', direction = 'LR')
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr3["lr3"]
        lr3_dummy[ ]
        style lr3_dummy fill:none,stroke:none
    end
    style node_lr3 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_dummy[ ]
        style ohe_dummy fill:none,stroke:none
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_pca["pca"]
        pca_dummy[ ]
        style pca_dummy fill:none,stroke:none
    end
    style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_dummy[ ]
        style std_dummy fill:none,stroke:none
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr3
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr3
    node_pca --> node_lr3
    node_std --> node_pca
```

**Path from Root to 'lr3' (3 path(s) found)**

In [28]:
Markdown(
    e.desc_node('lr3', direction = 'LR', show_params=True)
)

```mermaid
graph LR

    Root([Root])
    style Root fill:#fff9c4,stroke:#f57c00,stroke-width:3px

    subgraph node_lr3["lr3"]
        lr3_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>LogisticRegression</td></tr><tr><td align='left'><b>method</b></td><td align='left'>predict_proba</td></tr>"]
    end
    style node_lr3 fill:#ffcdd2,stroke:#c62828,stroke-width:3px

    subgraph node_ohe["ohe"]
        ohe_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>OneHotEncoder</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr><tr><td align='left'><b>sparse_output</b></td><td align='left'>False</td></tr></table>"]
    end
    style node_ohe fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_pca["pca"]
        pca_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>PCA</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr><tr><td align='left'><b>n_components</b></td><td align='left'>0.9</td></tr></table>"]
    end
    style node_pca fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    subgraph node_std["std"]
        std_info["<table><tr><td align='left'><b>processor</b></td><td align='left'>StandardScaler</td></tr><tr><td align='left'><b>method</b></td><td align='left'>transform</td></tr>"]
    end
    style node_std fill:#c8e6c9,stroke:#388e3c,stroke-width:2px

    Root --> node_lr3
    Root --> node_ohe
    Root --> node_std
    node_ohe --> node_lr3
    node_pca --> node_lr3
    node_std --> node_pca
```

**Path from Root to 'lr3' (3 path(s) found)**

In [29]:
e.add_grp('cb', parent_grp='clf', processor=cb.CatBoostClassifier, params={'verbose': 0})

In [30]:
e.set_node('cb1',  grp = 'cb', edges = [(None, X_num), (None, X_cat)], params = {'cat_features': X_cat})

In [31]:
import lightgbm as lgb

In [32]:
e.add_grp('lgb', parent_grp='clf', processor=lgb.LGBMClassifier, params={'verbose': -1})

In [33]:
e.set_node('lgb1',  grp = 'lgb', edges = [(None, X_num), (None, X_cat)], params={'categorical_features': X_cat})

In [34]:
e.exp(None)

🔄 Experimenting 5 node(s)
0 fold
  ├─ Experimenting 'lr1'...
  ├─ Experimenting 'cb1'...
  ├─ Experimenting 'lgb1'...
  ├─ Experimenting 'lr3'...
  ├─ Experimenting 'lr2'...
1 fold
  ├─ Experimenting 'lr1'...
  ├─ Experimenting 'cb1'...
  ├─ Experimenting 'lgb1'...
  ├─ Experimenting 'lr3'...
  ├─ Experimenting 'lr2'...
2 fold
  ├─ Experimenting 'lr1'...
  ├─ Experimenting 'cb1'...
  ├─ Experimenting 'lgb1'...
  ├─ Experimenting 'lr3'...
  ├─ Experimenting 'lr2'...
✅ Experimentation complete!


In [35]:
from modeler._metric import Metric
from modeler._stacker import Stacker
m = Metric('AUC', e, [(None, y)], slice(-1, None), roc_auc_score, include_train = True)
m2 = Metric('AUC2', e, [(None, y)], slice(None, 1), roc_auc_score, include_train = True)
s = Stacker(e, [(None, y)], slice(-1, None))
s2 = Stacker(e, [(None, y)], slice(None, 1))

metrics = {'AUC': m, 'AUC2': m2}
stackers = {'S1': s, 'S2': s2}

In [36]:
exps = ['lr1', 'lr2', 'cb1', 'lgb1']

for exp in exps:
    result_iter = e.nodes[exp].start_experiment()
for v in metrics.values():
    for exp in exps:
        v._start(exp)
for v in stackers.values():
    for exp in exps:
        v._start(exp)
for i in range(e.get_n_splits()):
    target_metrics = {
        k: v._get_data(i) for k, v in metrics.items()
    }
    for exp in exps:
        result_iter = e.nodes[exp].experiment(i)
        stacks = {
            k: list() for k in stackers.keys()
        }
        sub_metrics = {
            k: list() for k in metrics.keys()
        }
        for n, result_data in enumerate(result_iter):
            for k, v in metrics.items():
                sub_metric = v._get_metric(target_metrics[k][n], result_data)
                sub_metric = {k_sub: v_sub for k_sub, v_sub in sub_metric.items()}
                sub_metrics[k].append(sub_metric)
            for k, v in stackers.items():
                stacks[k].append(
                    v._get_valid(result_data)
                )
        for k, v in metrics.items():
            v._set_metric(exp, i, sub_metrics[k])
        for k, v in stackers.items():
            stk = v._aggregate(iter(stacks[k]))
            v._stack(exp, i, stk)
for exp in exps:
    result_iter = e.nodes[exp].end_experiment()

for v in metrics.values():
    for exp in exps:
        v._end(exp)
for v in stackers.values():
    for exp in exps:
        v._end(exp)

  Progress: 100/100 (100.0%) | training-binary_logloss: 0.0820, valid_1-binary_logloss: 0.2943

/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


  Progress: 100/100 (100.0%) | training-binary_logloss: 0.0752, valid_1-binary_logloss: 0.2759

/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


  Progress: 100/100 (100.0%) | training-binary_logloss: 0.0738, valid_1-binary_logloss: 0.2509

/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")
/home/sun9sun9/python312/lib/python3.12/site-packages/lightgbm/basic.py:2142: UserWarning: categorical_features in param dict is overridden.
  _log_warning(f"{cat_alias} in param dict is overridden.")


In [37]:
metrics['AUC'].get_metrics('.*')

0                             1                             2  \
             0                             0                             0   
         valid train_sub valid_sub     valid train_sub valid_sub     valid   
lr1   0.777506  0.781213  0.764964  0.787993  0.771995  0.802939  0.768093   
lr2   0.923399  0.917750  0.905762  0.913135  0.918720  0.943896  0.912170   
cb1   0.925365  0.941166  0.911951  0.914850  0.938722  0.938626  0.907851   
lgb1  0.915489  0.999511  0.883920  0.898174  0.999886  0.906720  0.894105   

                          
                          
     train_sub valid_sub  
lr1   0.785705  0.783333  
lr2   0.923800  0.901250  
cb1   0.966129  0.914267  
lgb1  0.999585  0.917502

In [38]:
stackers['S2'].get_dataset(None)

,lr1__loan_paid_back_0,lr2__loan_paid_back_0,cb1__loan_paid_back_0,lgb1__loan_paid_back_0,loan_paid_back
id,,,,,
176836,0.433459,0.288399,0.473124,0.320840,1
321322,0.160347,0.089303,0.055815,0.030392,0
225907,0.343629,0.220159,0.459149,0.603950,0
290104,0.427014,0.290407,0.485556,0.338905,1
453658,0.172569,0.059074,0.065568,0.048429,1
...,...,...,...,...,...
425863,0.085778,0.035443,0.012797,0.006059,1
103194,0.059540,0.023199,0.028573,0.003195,1
252332,0.522531,0.988603,0.991881,0.994327,0


In [39]:
stackers['S1'].get_dataset(None)

,lr1__loan_paid_back_1,lr2__loan_paid_back_1,cb1__loan_paid_back_1,lgb1__loan_paid_back_1,loan_paid_back
id,,,,,
176836,0.566541,0.711601,0.526876,0.679160,1
321322,0.839653,0.910697,0.944185,0.969608,0
225907,0.656371,0.779841,0.540851,0.396050,0
290104,0.572986,0.709593,0.514444,0.661095,1
453658,0.827431,0.940926,0.934432,0.951571,1
...,...,...,...,...,...
425863,0.914222,0.964557,0.987203,0.993941,1
103194,0.940460,0.976801,0.971427,0.996805,1
252332,0.477469,0.011397,0.008119,0.005673,0


In [41]:
e.desc_node_vars('pca', 0)

ValueError: Node 'pca' is not built yet. Please call node.build() first.

In [ ]:
e.nodes['cb1'].objs_[0][0][0].obj.evals_result_['validation_1']

In [ ]:
Markdown(
    e.desc_node('cb1', direction = 'LR')
)

In [ ]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

In [ ]:
Markdown(
    e.desc_node('lgb1', direction = 'LR')
)

In [ ]:
e._find_descendants('std')

In [ ]:
# e = Experimenter(df_train, sp = skf, sp_v = None, splitter_params = {'y': y})

In [ ]:
Markdown(
    e.desc_pipeline(max_depth=2, direction = 'LR')
)

In [ ]:
from modeler import create_like

In [ ]:
e2 = create_like(e, df_train.sample(frac = 0.01, random_state = 123), sp = ss, sp_v = StratifiedKFold(2, random_state=123, shuffle = True),
                splitter_params = {'y': y})

In [ ]:
e2.build()

In [ ]:
e2.nodes['lgb1'].get_result("evals_result")

In [ ]:
for (true_train, true_train_v), (prd_train, prd_train_v) in  zip(
    e.get_node_train_output(0, None, [y]),
    e.get_node_train_output(0, 'cb1', slice(-1, None))
):
    print(
        roc_auc_score(true_train.data, prd_train.data), 
        roc_auc_score(true_train_v.data, prd_train_v.data)
    )

In [ ]:
for true_valid, prd_valid in  zip(
    e.get_node_valid_output(0, None, [y]),
    e.get_node_valid_output(0, 'cb1', slice(-1, None))
):
    print(
        roc_auc_score(true_valid.data, prd_valid.data)
    )

In [ ]:
class Metric:
    def __init__(
        self, e, target_edge, output_var, metric_func, include_train = False
    ):
        self.e = e
        self.target_edge = target_edge
        self.output_var = output_var
        self.include_train = include_train
        self.metric_func = metric_func
        self.result = {}
        self.build_ids = {}

    def calc_idx(self, nodes, idx):
        result = {}
        grps = {}
        if self.include_train:
            for node in nodes:
                build_id = self.build_ids.get((node, idx), '')
                current_build_id = ''.join([i['build_id'] for _, _, i in self.e.nodes[node].objs_[idx]])
                if build_id == current_build_id and (node, idx) in self.result:
                    result[node] = self.result[(node, idx)]
                    grps[node] = e.get_parents(node)
                    continue
                iterator = zip(self.e.get_node_output(idx, None, [y]), self.e.get_node_output(idx, node, slice(-1, None)))
                self.build_ids[(node, idx)] = current_build_id
                for no, ((true_train, true_valid), (prd_train, prd_valid)) in enumerate(iterator):
                    result_train = self.metric_func(true_train[0].data, prd_train[0].data)
                    result_valid = self.metric_func(true_valid.data, prd_valid.data)
                    if true_train[1] is not None:
                        result_sub = {
                            (idx, 'train', f'train_{no}'): result_train,
                            (idx, 'train', f'valid_{no}'): self.metric_func(true_train[1].data, prd_train[1].data),
                            (idx, 'valid', ''): result_valid
                        }
                    else:
                        result_sub = {
                            (idx, 'train'): result_train, (idx, 'valid'): result_valid
                        }
                self.result[(node, idx)] = pd.Series(result_sub)
                result[node] = self.result[(node, idx)]
                grps[node] = e.get_parents(node)
        else:
            for node in nodes:
                build_id = self.build_ids.get((node, idx), '')
                current_build_id = ''.join([i['build_id'] for _, _, i in self.e.nodes[node].objs_[idx]])
                if build_id == current_build_id and node in self.result:
                    result[node] = self.result[(node, idx)]
                    grps[node] = e.get_parents(node)
                    continue
                iterator = zip(self.e.get_node_valid_output(idx, None, [y]), self.e.get_node_valid_output(idx, node, slice(-1, None)))
                self.build_ids[(node, idx)] = current_build_id
                for true_valid, prd_valid in iterator:
                    result_valid = self.metric_func(true_valid.data, prd_valid.data)                    
                    result_sub = {idx: result_valid}
                self.result[(node, idx)] = pd.Series(result_sub)
                result[node] = self.result[(node, idx)]
                grps[node] = e.get_parents(node)

        c, mx = None, -1
        for i in grps.values():
            i = i[::-1]
            if c is None:
                c = i
            else:
                mx = max(mx, len(i))
                for j in range(min(len(c), len(i))):
                    if c[j] != i[j]:
                        c = i[:j]
                        break
            if len(c) == 0:
                break
        for k, i in grps.items():
            i = tuple([''] * (mx - len(i) - len(c)) +  i[:-len(c)] + [k])
            result[k] = result[k].rename(i)
        return pd.DataFrame(result.values())

    def calc(self, nodes):
        result = [self.calc_idx(nodes, i) for i in range(self.e.get_n_splits())]
        return pd.concat(result, axis=1)

In [ ]:
for (y_true_train_t, y_true_valid_t), y_true_valid in e2.get_node_output(0, 'cb1'):
    print(y_true_train_t)
    print(y_true_valid_t)

In [ ]:
e.nodes['cb1'].objs_[0][0][0].X_

In [ ]:
import analyzer
from analyzer import Metric
importlib.reload(analyzer)

In [ ]:
m = Metric(e, (None, [y]), slice(-1, None), roc_auc_score, True)
m.set_nodes('clf')
result = m.get_metric()
result

In [ ]:
e3 = create_like(e, df_train.sample(frac = 0.01, random_state = 123), sp = skf, sp_v = None, splitter_params = {'y': y})

In [ ]:
df_input, df_output = e2.desc_node_vars('lr3', 0)
display(df_input)
df_output

In [ ]:
from analyzer import Stacker
s = Stacker(e3, (None, [y]), slice(-1, None))

In [ ]:
s.set_nodes('cb')
s.set_nodes('lr')
s.get_dataset().data

In [ ]:
e3.nodes['lr3'].objs_[0][0][0].obj.classes_

In [ ]:
lr_a.set_nodes('lr')

In [ ]:
for inner_idx, df in lr_a.result[('lr1', 0)].items():
    print(type(df['intercept']) == pd.Series, df['intercept'].to_frame().columns)

In [ ]:
lr_a.get_coef('lr1').T.groupby(level = [2]).mean().T

In [ ]:
lr_a.get_intercept('lr1').T.groupby(level = [2]).mean().T

In [ ]:
e.root.data

In [ ]:
for i in e2.get_node_valid_output(0, 'cb1', slice(0, -1)):
    print(i.data)

In [ ]:
e2.get_data_valid(0, [('lr1', slice(0, -1))])

In [ ]:
for i in e2.get_data_valid(0, [('lr1', slice(0, -1))]):
    print(i.data)

In [ ]:
e3 = create_like(e, df_train.sample(frac = 0.1, random_state = 123), splitter_params = {'y': y})

In [ ]:
e3.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])

In [ ]:
e3.add_grp('clf', edges = [(None, [y])], y = y, method = 'predict_proba')

In [ ]:
e3.add_grp('lr', parent_grp = 'clf', processor = LogisticRegression, edges = [('std', None)])

In [ ]:
e3.set_node('lr1', 'lr')

In [ ]:
p = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor({
        'loan_paid_back': pl.col('loan_paid_back').cast(pl.Int8)
    }),
    #sgpp.PandasConverter(index_col = 'id')
)
df_train = p.fit_transform('data/train.csv')
df_test = p.transform('data/test.csv')

In [ ]:
X_all = df_test.columns
X_num = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
X_cat = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
y = 'loan_paid_back'

In [ ]:
e = Experimenter(df_train, sp = skf, sp_v = ss_v, splitter_params = {'y': y})

In [ ]:
e.add_grp('clf', edges = [(None, [y])], y = y, method = 'predict_proba')
e.add_grp('preprocessor', method = 'transform')

In [ ]:
from sklearn.preprocessing import StandardScaler
e.set_node('std', 'preprocessor', StandardScaler, edges = [(None, X_num)])

In [ ]:
for a, b in e.get_data(0, [(None, X_num)]):
    print(a[0].data, b)

In [ ]:
from sklearn.linear_model import LogisticRegression

e.add_grp('lr', parent_grp = 'clf', processor = LogisticRegression, edges = [('std', None)])

In [ ]:
e.set_node('lr1', 'lr')

In [ ]:
for i in e.get_data_valid(0, [(None, y)]):
    print(i.data)

In [ ]:
ss = StratifiedKFold(n_splits=3)
for a, b in ss.split(df_train[X_all],  df_train[y]):
    pass

In [ ]:
lr = LogisticRegression()
lr.fit(df_train[X_num], df_train[[y]])

In [ ]:
df_train.to_pandas()[[y]].shape

In [ ]:
e.nodes['lr1'].y